# RAG Multimodal de Histologia
**Grupo 2 | CETEC UBATIC 2026-27**

## Orden de ejecucion
| Celda | Descripcion | Repetir? |
|-------|-------------|----------|
| 1 | Instalar paquetes base | Solo 1 vez por sesion |
| 2 | Instalar paquetes adicionales | Solo 1 vez por sesion |
| 3 | Credenciales y variables globales | Solo 1 vez por sesion |
| 4 | Imports y constantes | Solo 1 vez por sesion |
| 5 | Definicion de clases auxiliares | Solo 1 vez por sesion |
| 6 | Definicion de AsistenteHistologiaMultimodal | Solo 1 vez por sesion |
| 7 | Inicializar componentes (carga CLIP + LLM) | Solo 1 vez por sesion |
| 8 | Procesar PDFs y extraer temario | Solo 1 vez por sesion |
| 9 | Indexar en Qdrant | Saltear si ya esta indexado |
| 10 | Consulta interactiva | Cada vez que quieras consultar |

## Celda 1 — Instalar paquetes base

In [1]:
!pip install google-generativeai torch numpy PyPDF2 --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.6/232.6 kB 8.6 MB/s eta 0:00:00


## Celda 2 — Instalar paquetes adicionales

In [2]:
!pip install torch torchvision torchaudio \
    langchain langgraph langchain_google_genai \
    ragas sentence-transformers transformers \
    pdf2image Pillow pdfplumber pytesseract \
    qdrant-client \
    langchain-community langsmith \
    langgraph-checkpoint --quiet

!apt-get update -qq && apt-get install -y -q poppler-utils tesseract-ocr
print('Paquetes adicionales listos')

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 466.5/466.5 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 71.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 398.1/398.1 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 74.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 33.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 26.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 78.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 k

## Celda 3 — Credenciales y variables globales
> Requiere los siguientes secretos en Colab:
> `GOOGLE_API_KEY`, `QDRANT_URL`, `QDRANT_KEY`, `HF_TOKEN`, `LANGSMITH_API_KEY` (opcional)

In [4]:
import os

try:
    from google.colab import userdata
    COLAB_ENV = True
except ImportError:
    COLAB_ENV = False
    class userdata:
        @staticmethod
        def get(key):
            return os.getenv(key)

os.environ['GOOGLE_API_KEY'] = userdata.get('GOOGLE_API_KEY') or ''

# LangSmith (opcional)
def _configurar_langsmith():
    api_key = userdata.get('LANGSMITH_API_KEY')
    if not api_key:
        print('LangSmith: LANGSMITH_API_KEY no encontrada - tracing desactivado')
        return False
    project = userdata.get('LANGSMITH_PROJECT')
    os.environ['LANGCHAIN_TRACING_V2'] = 'true'
    os.environ['LANGCHAIN_API_KEY']    = api_key
    os.environ['LANGCHAIN_PROJECT']    = project
    print(f'LangSmith activado -> proyecto: {project}')
    return True

LANGSMITH_ACTIVO = _configurar_langsmith()

# Constantes globales
SIMILARITY_THRESHOLD    = 0.22
DIRECTORIO_IMAGENES     = '/content/imagenes_extraidas'
DIRECTORIO_PDFS         = '/content/pdf'
CHUNK_SIZE              = 500

FEATURES_DISCRIMINATORIAS = [
    'presencia/ausencia de lumen central',
    'estratificacion celular (capas concentricas vs difusa)',
    'tipo de queratinizacion (parakeratosis, ortoqueratosis, ninguna)',
    'aspecto del nucleo (picnotico, fantasma, ausente, vesicular)',
    'celulas fantasma (si/no)',
    'material amorfo central (si/no y aspecto)',
    'patron de tincion HE (eosinofilia, basofilia)',
    'tamano estimado de la estructura',
    'tejido circundante (estroma, epitelio, piel, otro)',
    'reaccion inflamatoria perilesional (si/no, tipo)',
]

ANCLAS_SEMANTICAS_HISTOLOGIA = [
    'histologia tejido celular microscopia',
    'tipos de tejido epitelial conectivo muscular nervioso',
    'coloracion hematoxilina eosina HE tincion histologica',
    'estructuras celulares nucleo citoplasma membrana',
    'diagnostico diferencial patologia biopsia',
    'glandulas epitelio estratificado cilindrico simple',
    'identificar tejido muestra microscopica',
    'que tipo de tejido es este',
    'cual es la estructura observada en la imagen',
    'clasificar celula estructura histologica',
    'tumor quiste foliculo cuerpo luteo albicans',
    'corte histologico preparacion muestra lamina',
]

print('Credenciales y constantes cargadas')
print(f'  SIMILARITY_THRESHOLD : {SIMILARITY_THRESHOLD}')
print(f'  DIRECTORIO_PDFS      : {DIRECTORIO_PDFS}')
print(f'  CHUNK_SIZE           : {CHUNK_SIZE}')

LangSmith activado -> proyecto: CETEC_g2
Credenciales y constantes cargadas
  SIMILARITY_THRESHOLD : 0.22
  DIRECTORIO_PDFS      : /content/pdf
  CHUNK_SIZE           : 500


## Celda 4 — Imports generales y utilidades

In [5]:
import json, time, asyncio, re, glob
import nest_asyncio
import torch
import numpy as np
from typing import TypedDict, Annotated, List, Dict, Any, Optional
from PIL import Image
import base64
from PyPDF2 import PdfReader
from pdf2image import convert_from_path
import pytesseract
import operator

from transformers import CLIPProcessor, CLIPModel, CLIPTokenizer
from qdrant_client import AsyncQdrantClient, models
from qdrant_client.models import PointStruct, VectorParams, Distance

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver

nest_asyncio.apply()

def _safe(value, default: str = '') -> str:
    return value if isinstance(value, str) and value else default

print('Imports OK')
print(f'  PyTorch : {torch.__version__}')
print(f'  CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'  GPU     : {torch.cuda.get_device_name()}')

/usr/local/lib/python3.12/dist-packages/langgraph/cache/base/__init__.py:8: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


Imports OK
  PyTorch : 2.10.0+cu128
  CUDA    : True
  GPU     : Tesla T4


## Celda 5 — Clases auxiliares
Define: `AgentState`, `SemanticMemory`, `ClasificadorSemantico`,
`ExtractorImagenesPDF`, `ExtractorTemario`

In [6]:
# ── AgentState ───────────────────────────────────────────────────────────
class AgentState(TypedDict):
    messages:                    Annotated[list, operator.add]
    consulta_texto:              str
    imagen_path:                 Optional[str]
    imagen_embedding:            Optional[List[float]]
    contexto_memoria:            str
    contenido_base:              str
    clasificacion:               str
    consulta_busqueda_texto:     str
    consulta_busqueda_visual:    str
    resultados_busqueda:         List[Dict[str, Any]]
    resultados_validos:          List[Dict[str, Any]]
    contexto_documentos:         str
    respuesta_final:             str
    trayectoria:                 Annotated[List[Dict[str, Any]], operator.add]
    user_id:                     str
    tiempo_inicio:               float
    analisis_visual:             Optional[str]
    tiene_imagen:                bool
    imagen_es_nueva:             bool
    contexto_suficiente:         bool
    temario:                     List[str]
    tema_valido:                 bool
    tema_encontrado:             Optional[str]
    imagenes_recuperadas:        List[str]
    analisis_comparativo:        Optional[str]
    estructura_identificada:     Optional[str]
    similitud_semantica_dominio: float


# ── SemanticMemory ────────────────────────────────────────────────────────
class SemanticMemory:
    def __init__(self, llm, max_entries: int = 10):
        self.llm              = llm
        self.conversations    = []
        self.max_entries      = max_entries
        self.summary          = ''
        self.direct_history   = ''
        self.imagen_activa_path: Optional[str] = None
        self.imagen_turno_subida: int = 0
        self.turno_actual: int = 0

    def set_imagen(self, path: Optional[str]):
        if path and os.path.exists(path):
            self.imagen_activa_path  = path
            self.imagen_turno_subida = self.turno_actual
            print(f'   Imagen activa: {path}')
        elif path is None:
            self.imagen_activa_path = None
            print('   Imagen activa limpiada')

    def get_imagen_activa(self) -> Optional[str]:
        if self.imagen_activa_path and os.path.exists(self.imagen_activa_path):
            return self.imagen_activa_path
        return None

    def tiene_imagen_previa(self) -> bool:
        return self.get_imagen_activa() is not None

    def add_interaction(self, query: str, response: str):
        self.turno_actual += 1
        self.conversations.append({
            'query': query, 'response': response,
            'turno': self.turno_actual, 'imagen': self.imagen_activa_path
        })
        if len(self.conversations) > self.max_entries:
            self.conversations.pop(0)
        self.direct_history += f'\nUsuario: {query}\nAsistente: {response}\n'
        if len(self.conversations) > 3:
            recent = self.conversations[-3:]
            self.direct_history = ''
            for conv in recent:
                img_nota = f' [imagen: {os.path.basename(conv["imagen"])}]' \
                           if conv.get('imagen') else ''
                self.direct_history += (
                    f'\nUsuario{img_nota}: {conv["query"]}'
                    f'\nAsistente: {conv["response"]}\n'
                )
        self._update_summary()

    def _update_summary(self):
        try:
            if len(self.conversations) > 6:
                resp = self.llm.invoke([
                    SystemMessage(content='Resume estas interacciones medicas manteniendo detalles tecnicos:'),
                    HumanMessage(content=self.direct_history)
                ])
                self.summary = f'Resumen: {resp.content}\n\nRecientes:{self.direct_history}'
            else:
                self.summary = f'Recientes:{self.direct_history}'
        except Exception as e:
            print(f'Error actualizando resumen: {e}')
            self.summary = f'Recientes:{self.direct_history}'

    def get_context(self) -> str:
        ctx = self.summary.strip() or 'No hay conversacion previa.'
        if self.imagen_activa_path:
            ctx += f'\n\n[Imagen activa: {os.path.basename(self.imagen_activa_path)}]'
        return ctx


# ── ClasificadorSemantico ─────────────────────────────────────────────────
class ClasificadorSemantico:
    UMBRAL_SIMILITUD = 0.18
    UMBRAL_LLM       = 0.5

    def __init__(self, llm, clip_model, clip_tokenizer, device, temario):
        self.llm              = llm
        self.clip_model       = clip_model
        self.clip_tokenizer   = clip_tokenizer
        self.device           = device
        self.temario          = temario
        self._anclas_emb: Optional[np.ndarray] = None

    def _embed_texto(self, textos):
        inputs = self.clip_tokenizer(
            textos, padding=True, truncation=True,
            max_length=77, return_tensors='pt'
        ).to(self.device)
        with torch.no_grad():
            feats = self.clip_model.get_text_features(**inputs)
            emb   = feats.pooler_output.cpu().numpy() \
                    if hasattr(feats, 'pooler_output') \
                    else feats.cpu().numpy()
        norms = np.linalg.norm(emb, axis=1, keepdims=True)
        return emb / np.maximum(norms, 1e-9)

    def _get_anclas_emb(self):
        if self._anclas_emb is None:
            print('   Precalculando embeddings de anclas...')
            self._anclas_emb = self._embed_texto(ANCLAS_SEMANTICAS_HISTOLOGIA)
        return self._anclas_emb

    def similitud_con_dominio(self, consulta: str) -> float:
        try:
            q_emb = self._embed_texto([consulta])
            a_emb = self._get_anclas_emb()
            sims  = (q_emb @ a_emb.T).flatten()
            return float(np.max(sims))
        except Exception as e:
            print(f'   Error similitud: {e}')
            return 0.0

    async def clasificar(self, consulta, analisis_visual=None,
                          imagen_activa=False, temario_muestra=None):
        sim = self.similitud_con_dominio(consulta)
        umbral_efectivo = self.UMBRAL_SIMILITUD * (0.6 if imagen_activa else 1.0)
        if sim >= umbral_efectivo:
            return {'valido': True, 'tema_encontrado': None,
                    'motivo': f'CLIP sim={sim:.3f}', 'similitud_dominio': sim,
                    'metodo': 'semantico_clip'}
        muestra_temas = (temario_muestra or self.temario)[:60]
        temario_txt   = '\n'.join(f'- {t}' for t in muestra_temas)
        context_extra = ''
        if analisis_visual:
            context_extra = f'\nANALISIS VISUAL:\n{analisis_visual[:600]}'
        if imagen_activa:
            context_extra += '\n[Hay imagen histologica activa en el chat]'
        system = (
            'Eres un clasificador de intencion para un sistema RAG de histologia medica.\n'
            'Determina si la consulta es sobre histologia, patologia o morfologia celular.\n'
            'IMPORTANTE: preguntas como que tejido es este son histologicas aunque no\n'
            'usen terminos tecnicos. Si hay imagen activa, dar beneficio de la duda.\n\n'
            f'TEMARIO (muestra):\n{temario_txt}\n{context_extra}\n\n'
            'Responde SOLO en JSON valido (sin backticks):\n'
            '{"valido": true/false, "tema_encontrado": "tema o null", '
            '"confianza": 0.0-1.0, "motivo": "explicacion breve"}'
        )
        try:
            resp  = await self.llm.ainvoke([
                SystemMessage(content=system),
                HumanMessage(content=f'CONSULTA: {consulta}')
            ])
            texto = re.sub(r'```json\s*|\s*```', '', resp.content.strip())
            data  = json.loads(texto)
            valido = bool(data.get('valido', True))
            if not valido and imagen_activa and float(data.get('confianza',0.5)) < 0.7:
                valido = True
                data['motivo'] += ' [aceptado por imagen activa]'
            return {'valido': valido, 'tema_encontrado': data.get('tema_encontrado'),
                    'motivo': data.get('motivo',''), 'similitud_dominio': sim,
                    'metodo': 'llm' if sim < umbral_efectivo*0.5 else 'combinado'}
        except Exception as e:
            return {'valido': imagen_activa or sim > 0.10,
                    'tema_encontrado': None, 'motivo': f'Fallback: {e}',
                    'similitud_dominio': sim, 'metodo': 'fallback'}


# ── ExtractorImagenesPDF ──────────────────────────────────────────────────
class ExtractorImagenesPDF:
    RENDER_DPI = 150

    def __init__(self, directorio_salida=DIRECTORIO_IMAGENES):
        self.directorio_salida = directorio_salida
        os.makedirs(directorio_salida, exist_ok=True)

    def extraer_de_pdf(self, pdf_path):
        imagenes   = []
        nombre_pdf = os.path.splitext(os.path.basename(pdf_path))[0]
        try:
            paginas = convert_from_path(pdf_path, dpi=self.RENDER_DPI)
        except Exception as e:
            print(f'Error renderizando {pdf_path}: {e}')
            return []
        for num_pagina, pil_img in enumerate(paginas, start=1):
            nombre_archivo = f'{nombre_pdf}_pag{num_pagina}.png'
            ruta_completa  = os.path.join(self.directorio_salida, nombre_archivo)
            try:
                pil_img.save(ruta_completa, format='PNG')
                try:
                    ocr_text = pytesseract.image_to_string(pil_img).strip()[:300]
                except:
                    ocr_text = ''
                imagenes.append({
                    'path': ruta_completa, 'fuente_pdf': os.path.basename(pdf_path),
                    'pagina': num_pagina, 'indice': 1, 'ocr_text': ocr_text
                })
            except Exception as e:
                print(f'  Error guardando pagina {num_pagina}: {e}')
        print(f'  {len(imagenes)} paginas extraidas de {os.path.basename(pdf_path)}')
        return imagenes

    def extraer_de_directorio(self, directorio):
        todas = []
        pdfs  = glob.glob(os.path.join(directorio, '*.pdf'))
        print(f'Extrayendo paginas de {len(pdfs)} PDFs...')
        for pdf_path in pdfs:
            todas.extend(self.extraer_de_pdf(pdf_path))
        print(f'Total imagenes extraidas: {len(todas)}')
        return todas


# ── ExtractorTemario ──────────────────────────────────────────────────────
class ExtractorTemario:
    def __init__(self, llm):
        self.llm   = llm
        self.temas = []

    async def extraer_temario(self, texto_completo):
        print('Extrayendo temario con LLM...')
        muestra = texto_completo[:8000]
        system  = (
            'Eres un experto en histologia medica. Analiza el texto y genera una lista '
            'EXHAUSTIVA de todos los temas, estructuras, tejidos, celulas, tiniciones y '
            'patologias cubiertos.\n\n'
            'INSTRUCCIONES:\n'
            '- Lista cada tema en una linea separada\n'
            '- Se especifico (ej: epitelio cilindrico simple)\n'
            '- Incluye sinonimos comunes\n'
            '- Responde SOLO con la lista, sin introduccion ni bullets'
        )
        try:
            resp = await self.llm.ainvoke([
                SystemMessage(content=system),
                HumanMessage(content=f'TEXTO:\n{muestra}')
            ])
            temas_raw  = resp.content.strip().split('\n')
            self.temas = [t.strip() for t in temas_raw
                          if t.strip() and len(t.strip()) > 2]
            print(f'Temario: {len(self.temas)} temas')
            with open('temario_histologia.json', 'w', encoding='utf-8') as f:
                json.dump(self.temas, f, ensure_ascii=False, indent=2)
            return self.temas
        except Exception as e:
            print(f'Error extrayendo temario: {e}')
            return []

    def get_temario_texto(self):
        if not self.temas:
            return 'Temario no disponible.'
        return '\n'.join(f'- {t}' for t in self.temas[:100])


print('Clases auxiliares definidas')

Clases auxiliares definidas


## Celda 6 — Clase principal: AsistenteHistologiaMultimodal

In [7]:
class AsistenteHistologiaMultimodal:

    SIMILARITY_THRESHOLD = SIMILARITY_THRESHOLD

    def __init__(self):
        self.llm                  = None
        self.memoria              = None
        self.graph                = None
        self.compiled_graph       = None
        self.memory_saver         = None
        self.contenido_base       = ''
        self.clip_model_name      = 'openai/clip-vit-base-patch32'
        self.clip_model           = None
        self.clip_processor       = None
        self.clip_tokenizer       = None
        self.max_token_length     = 77
        self.qdrant_url           = userdata.get('QDRANT_URL')
        self.qdrant_api_key       = userdata.get('QDRANT_KEY')
        self.conch_model          = None
        self.conch_transform      = None
        self.collection_name      = 'histologia_v4_multimodal'
        self.extractor_imagenes   = ExtractorImagenesPDF(DIRECTORIO_IMAGENES)
        self.extractor_temario    = None
        self.clasificador_semantico = None
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'
        print('AsistenteHistologiaMultimodal v4.2 creado')

    # ── Inicializacion ────────────────────────────────────────────────────
    def inicializar_componentes(self):
        self._init_modelos()
        self.memoria              = SemanticMemory(llm=self.llm)
        self.extractor_temario    = ExtractorTemario(llm=self.llm)
        self._init_clip() # solo para texto
        self._init_conch() # encoder visual
        self.clasificador_semantico = ClasificadorSemantico(
            llm=self.llm, clip_model=self.clip_model,
            clip_tokenizer=self.clip_tokenizer,
            device=self.device, temario=[]
        )
        self.memory_saver   = MemorySaver()
        self._crear_grafo()
        self.compiled_graph = self.graph.compile(checkpointer=self.memory_saver)
        print('Todos los componentes inicializados')

    def _init_modelos(self):
        self.llm = ChatGoogleGenerativeAI(
            model='gemini-2.5-flash',
            google_api_key=userdata.get('GOOGLE_API_KEY'),
            temperature=0, max_output_tokens=None,
        )
        print('LLM inicializado (Gemini 2.5 Flash)')

    def _init_clip(self):
        print('Cargando CLIP...')
        self.clip_model     = CLIPModel.from_pretrained(self.clip_model_name).to(self.device)
        self.clip_processor = CLIPProcessor.from_pretrained(self.clip_model_name)
        self.clip_tokenizer = CLIPTokenizer.from_pretrained(self.clip_model_name)
        self.clip_model.eval()
        print(f'CLIP en {self.device}')

    def _init_conch(self):
        print('Cargando CONCH...')
        HF_TOKEN = userdata.get('HF_TOKEN')
        if not HF_TOKEN:
            raise ValueError('HF_TOKEN no encontrado en Colab secrets. '
                            'Agregalo en el panel de Secrets antes de continuar.')

        # Login HuggingFace
        if 'HF_TOKEN' in os.environ:
            del os.environ['HF_TOKEN']
        import timm
        from huggingface_hub import login, hf_hub_download
        login(token=HF_TOKEN, add_to_git_credential=False)
        os.environ['HF_TOKEN'] = HF_TOKEN

        # Descargar y cargar checkpoint
        checkpoint_path = hf_hub_download(
            repo_id='MahmoodLab/conch',
            filename='pytorch_model.bin',
            token=HF_TOKEN
        )
        checkpoint = torch.load(checkpoint_path, map_location='cpu')
        print(f'  Checkpoint OK | keys totales: {len(checkpoint)}')

        # Crear modelo ViT con timm (sin cabeza de clasificación)
        self.conch_model = timm.create_model(
            'vit_base_patch16_224',
            pretrained=False,
            num_classes=0
        )

        # Extraer solo los pesos visuales
        visual_sd = {
            k.replace('visual.', ''): v
            for k, v in checkpoint.items()
            if k.startswith('visual.')
        }
        visual_sd.pop('proj_contrast', None)

        missing, unexpected = self.conch_model.load_state_dict(visual_sd, strict=False)
        print(f'  Missing: {len(missing)} | Unexpected: {len(unexpected)}')
        # Missing esperado: 0 o muy pocos. Unexpected esperado: 0.
        # Si missing > 5, algo salió mal con el mapeo de keys.

        self.conch_model = self.conch_model.to(self.device)
        self.conch_model.eval()

        # Transform ImageNet estándar (igual que en la evaluación)
        from torchvision import transforms
        self.conch_transform = transforms.Compose([
            transforms.Resize(224),
            transforms.CenterCrop(224),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                std=[0.229, 0.224, 0.225]),
        ])

        # Smoke test
        from PIL import Image as PILImage
        test_emb = self._embed_imagen_pil(PILImage.new('RGB', (224, 224)))
        print(f'  Smoke test OK | dim={len(test_emb)} | '
              f'norma={round(float(np.linalg.norm(test_emb)), 4)}')
        # dim debe ser 768, norma debe ser 1.0000
        print(f'CONCH en {self.device}')

    # ── Grafo LangGraph ───────────────────────────────────────────────────
    def _crear_grafo(self):
        g = StateGraph(AgentState)
        g.add_node('inicializar',          self._nodo_inicializar)
        g.add_node('procesar_imagen',      self._nodo_procesar_imagen)
        g.add_node('clasificar',           self._nodo_clasificar)
        g.add_node('generar_consulta',     self._nodo_generar_consulta)
        g.add_node('buscar_qdrant',        self._nodo_buscar_qdrant)
        g.add_node('filtrar_contexto',     self._nodo_filtrar_contexto)
        g.add_node('analisis_comparativo', self._nodo_analisis_comparativo)
        g.add_node('generar_respuesta',    self._nodo_generar_respuesta)
        g.add_node('finalizar',            self._nodo_finalizar)
        g.add_node('fuera_temario',        self._nodo_fuera_temario)
        g.add_edge(START,                  'inicializar')
        g.add_edge('inicializar',          'procesar_imagen')
        g.add_edge('procesar_imagen',      'clasificar')
        g.add_conditional_edges(
            'clasificar', self._route_por_temario,
            {'en_temario': 'generar_consulta', 'fuera_temario': 'fuera_temario'}
        )
        g.add_edge('fuera_temario',        'finalizar')
        g.add_edge('generar_consulta',     'buscar_qdrant')
        g.add_edge('buscar_qdrant',        'filtrar_contexto')
        g.add_edge('filtrar_contexto',     'analisis_comparativo')
        g.add_edge('analisis_comparativo', 'generar_respuesta')
        g.add_edge('generar_respuesta',    'finalizar')
        g.add_edge('finalizar',            END)
        self.graph = g

    def _route_por_temario(self, state):
        return 'en_temario' if state.get('tema_valido', True) else 'fuera_temario'

    # ── Nodos ─────────────────────────────────────────────────────────────
    async def _nodo_inicializar(self, state):
        state['contexto_memoria']            = self.memoria.get_context()
        state['contenido_base']              = self.contenido_base
        state['tiempo_inicio']               = time.time()
        state['tiene_imagen']                = False
        state['imagen_es_nueva']             = False
        state['contexto_suficiente']         = False
        state['resultados_validos']          = []
        state['imagenes_recuperadas']        = []
        state['tema_valido']                 = True
        state['tema_encontrado']             = None
        state['temario']                     = self.extractor_temario.temas if self.extractor_temario else []
        state['analisis_comparativo']        = None
        state['estructura_identificada']     = None
        state['similitud_semantica_dominio'] = 0.0
        if not state.get('trayectoria'):
            state['trayectoria'] = []
        state['trayectoria'].append({'nodo': 'Inicializar', 'tiempo': 0})
        return state

    async def _nodo_procesar_imagen(self, state):
        t0 = time.time()
        imagen_path_nuevo = state.get('imagen_path')
        imagen_es_nueva   = False
        if imagen_path_nuevo and os.path.exists(imagen_path_nuevo):
            imagen_path_activo = imagen_path_nuevo
            imagen_es_nueva    = True
            self.memoria.set_imagen(imagen_path_activo)
        elif self.memoria.tiene_imagen_previa():
            imagen_path_activo = self.memoria.get_imagen_activa()
            state['imagen_path'] = imagen_path_activo
        else:
            imagen_path_activo = None
        if imagen_path_activo and os.path.exists(imagen_path_activo):
            try:
                emb = self._embed_imagen(imagen_path_activo)
                state['imagen_embedding'] = emb.tolist()
                state['tiene_imagen']     = True
                state['imagen_es_nueva']  = imagen_es_nueva
                if imagen_es_nueva or not state.get('analisis_visual'):
                    state['analisis_visual'] = await self._analizar_imagen_gemini(imagen_path_activo)
            except Exception as e:
                print(f'Error procesando imagen: {e}')
                state['imagen_embedding'] = None
                state['analisis_visual']  = None
                state['tiene_imagen']     = False
        else:
            state['imagen_embedding'] = None
            state['analisis_visual']  = None
            state['tiene_imagen']     = False
            state['imagen_es_nueva']  = False
        state['trayectoria'].append({'nodo': 'ProcesarImagen',
            'tiene_imagen': state['tiene_imagen'], 'tiempo': round(time.time()-t0,2)})
        return state

    async def _analizar_imagen_gemini(self, imagen_path):
        try:
            with open(imagen_path, 'rb') as f:
                data = base64.b64encode(f.read()).decode('utf-8')
            ext  = os.path.splitext(imagen_path)[1].lower()
            mime = 'image/png' if ext == '.png' else 'image/jpeg'
            features_lista = '\n'.join(
                f'  {i+1}. {f}' for i,f in enumerate(FEATURES_DISCRIMINATORIAS))
            msg = HumanMessage(content=[
                {'type': 'text', 'text': (
                    'Eres un patologo experto. Analiza esta imagen histologica.\n\n'
                    '**PARTE 1 - DESCRIPCION GENERAL:**\n'
                    'Tipo de tejido, coloracion, aumento estimado, estructuras principales.\n\n'
                    '**PARTE 2 - FEATURES DISCRIMINATORIAS:**\n'
                    f'{features_lista}\n\n'
                    '**PARTE 3 - DIAGNOSTICO DIFERENCIAL VISUAL:**\n'
                    'Las 3 estructuras mas probables, ordenadas por probabilidad.'
                )},
                {'type': 'image_url',
                 'image_url': {'url': f'data:{mime};base64,{data}'}}
            ])
            resp = await self.llm.ainvoke([msg])
            return resp.content
        except Exception as e:
            print(f'Analisis visual fallido: {e}')
            return ''

    async def _nodo_clasificar(self, state):
        t0 = time.time()
        system_clasif = (
            'Eres un patologo experto. Clasifica la consulta histologica.\n'
            'Formato:\nTEJIDO: [tipo]\nESTRUCTURAS: [estructuras]\n'
            'COLORACION: [tincion]\nKEYWORDS: [palabras clave]\n'
            'DIAGNOSTICO_POTENCIAL: [si aplica]'
        )
        user_parts = [
            f'CONSULTA: {state["consulta_texto"]}',
            f'HISTORIAL: {_safe(state.get("contexto_memoria"),"Sin historial")[:500]}'
        ]
        if _safe(state.get('analisis_visual')):
            user_parts.append(f'ANALISIS VISUAL: {state["analisis_visual"][:500]}')
        try:
            resp = await self.llm.ainvoke([
                SystemMessage(content=system_clasif),
                HumanMessage(content='\n\n'.join(user_parts))
            ])
            state['clasificacion'] = resp.content
        except Exception as e:
            state['clasificacion'] = 'sin clasificacion'
        verificacion = await self.clasificador_semantico.clasificar(
            consulta=state['consulta_texto'],
            analisis_visual=state.get('analisis_visual'),
            imagen_activa=state.get('tiene_imagen', False),
            temario_muestra=state.get('temario',[])[:60],
        )
        state['tema_valido']                 = verificacion.get('valido', True)
        state['tema_encontrado']             = verificacion.get('tema_encontrado')
        state['similitud_semantica_dominio'] = verificacion.get('similitud_dominio', 0.0)
        state['trayectoria'].append({'nodo': 'Clasificar',
            'tema_valido': state['tema_valido'], 'tiempo': round(time.time()-t0,2)})
        return state

    async def _nodo_fuera_temario(self, state):
        temario_resumen = '\n'.join(
            f'  - {t}' for t in (state.get('temario') or [])[:20])
        state['respuesta_final'] = (
            'Consulta fuera del dominio disponible.\n\n'
            f'Temas disponibles (muestra):\n{temario_resumen}\n\n'
            'Subi una imagen histologica y reformula tu pregunta.'
        )
        state['contexto_suficiente'] = False
        state['trayectoria'].append({'nodo': 'FueraTemario'})
        return state

    async def _nodo_generar_consulta(self, state):
        t0 = time.time()
        tema_extra = f'\nTEMA DETECTADO: {state["tema_encontrado"]}' \
                     if state.get('tema_encontrado') else ''
        system = (
            'Genera consultas de busqueda MUY CORTAS (max 8 palabras) '
            'para base de datos histologica.\n'
            'Formato OBLIGATORIO:\n'
            'CONSULTA_TEXTO: <texto>\nCONSULTA_VISUAL: <texto>'
        )
        user = (
            f'CLASIFICACION:\n{_safe(state.get("clasificacion"),"N/A")}'
            f'{tema_extra}\n\nCONSULTA ORIGINAL:\n{state["consulta_texto"]}'
        )
        if _safe(state.get('analisis_visual')):
            user += f'\n\nANALISIS VISUAL:\n{state["analisis_visual"][:400]}'
        try:
            resp     = await self.llm.ainvoke([SystemMessage(content=system),
                                               HumanMessage(content=user)])
            contenido = resp.content
            ct = state['consulta_texto'][:77]
            cv = ct
            if 'CONSULTA_TEXTO:' in contenido:
                partes = contenido.split('CONSULTA_TEXTO:')[1]
                if 'CONSULTA_VISUAL:' in partes:
                    ct = partes.split('CONSULTA_VISUAL:')[0].strip()[:77]
                    cv = partes.split('CONSULTA_VISUAL:')[1].strip()[:77]
                else:
                    ct = partes.strip()[:77]
        except:
            ct = cv = state['consulta_texto'][:77]
        state['consulta_busqueda_texto']  = ct
        state['consulta_busqueda_visual'] = cv
        state['trayectoria'].append({'nodo': 'GenerarConsulta', 'query': ct,
                                     'tiempo': round(time.time()-t0,2)})
        return state

    async def _nodo_buscar_qdrant(self, state):
        t0 = time.time()
        imagen_emb = state.get('imagen_embedding')
        tiene_emb  = isinstance(imagen_emb, list) and len(imagen_emb) > 0
        resultados_texto  = await self._search_qdrant(
            texto=state['consulta_busqueda_texto'], imagen_embedding=None, top_k=10)
        resultados_imagen = []
        if tiene_emb:
            resultados_imagen = await self._search_qdrant(
                texto=None, imagen_embedding=imagen_emb, top_k=10)
        combined = {}
        for r in resultados_texto + resultados_imagen:
            key = f'{r["fuente"]}_{r["texto"][:50]}'
            if key not in combined or r['similitud'] > combined[key]['similitud']:
                combined[key] = r
        state['resultados_busqueda'] = sorted(
            combined.values(), key=lambda x: x['similitud'], reverse=True)[:15]
        state['trayectoria'].append({'nodo': 'BuscarQdrant',
            'hits_texto': len(resultados_texto),
            'hits_imagen': len(resultados_imagen),
            'tiempo': round(time.time()-t0,2)})
        return state

    async def _nodo_filtrar_contexto(self, state):
        t0     = time.time()
        umbral = self.SIMILARITY_THRESHOLD
        validos = [r for r in state['resultados_busqueda']
                   if r['similitud'] >= umbral]
        state['resultados_validos']  = validos
        state['contexto_suficiente'] = len(validos) > 0
        vistas = set()
        imgs_unicas = []
        for r in validos:
            img_path = r.get('imagen_path')
            if img_path and os.path.exists(img_path) and img_path not in vistas:
                vistas.add(img_path)
                imgs_unicas.append(img_path)
        state['imagenes_recuperadas'] = imgs_unicas
        if validos:
            validos_sorted = sorted(validos, key=lambda x: x['similitud'], reverse=True)
            bloques = []
            for i, r in enumerate(validos_sorted, 1):
                enc = (f'[Fragmento {i} | Fuente: {r["fuente"]} | '
                       f'Tipo: {r["tipo"]} | Similitud: {r["similitud"]:.3f}')
                if r.get('imagen_path'):
                    enc += f' | Imagen: {os.path.basename(r["imagen_path"])}'
                enc += ']'
                bloques.append(f'{enc}\n{r["texto"][:600]}')
            state['contexto_documentos'] = '\n\n'.join(bloques)
        else:
            state['contexto_documentos'] = ''
        state['trayectoria'].append({'nodo': 'FiltrarContexto',
            'hits_validos': len(validos), 'tiempo': round(time.time()-t0,2)})
        return state

    async def _nodo_analisis_comparativo(self, state):
        t0 = time.time()
        if not state.get('tiene_imagen') or not state.get('imagen_path'):
            state['trayectoria'].append({'nodo': 'AnalisisComparativo',
                'motivo': 'sin imagen', 'tiempo': round(time.time()-t0,2)})
            return state
        imagenes_referencia = [
            p for p in state.get('imagenes_recuperadas',[])[:3]
            if os.path.exists(p)
        ]
        if not imagenes_referencia:
            state['analisis_comparativo'] = None
            state['trayectoria'].append({'nodo': 'AnalisisComparativo',
                'motivo': 'sin referencias', 'tiempo': round(time.time()-t0,2)})
            return state
        content_parts = [{'type': 'text', 'text': (
            'Eres un patologo experto. Compara la imagen de consulta '
            'con las imagenes de referencia.\n\n=== IMAGEN DE CONSULTA ===' )}]
        try:
            with open(state['imagen_path'], 'rb') as f:
                data_user = base64.b64encode(f.read()).decode('utf-8')
            ext  = os.path.splitext(state['imagen_path'])[1].lower()
            mime = 'image/png' if ext == '.png' else 'image/jpeg'
            content_parts.append({'type': 'image_url',
                'image_url': {'url': f'data:{mime};base64,{data_user}'}})
        except Exception as e:
            print(f'No se pudo cargar imagen del usuario: {e}')
            state['analisis_comparativo'] = None
            return state
        if _safe(state.get('analisis_visual')):
            content_parts.append({'type': 'text',
                'text': f'\nAnalisis previo:\n{state["analisis_visual"][:600]}\n'})
        for i, ref_path in enumerate(imagenes_referencia, 1):
            content_parts.append({'type': 'text',
                'text': f'\n=== IMAGEN DE REFERENCIA #{i} ({os.path.basename(ref_path)}) ==='})
            try:
                with open(ref_path, 'rb') as f:
                    data_ref = base64.b64encode(f.read()).decode('utf-8')
                ext  = os.path.splitext(ref_path)[1].lower()
                mime = 'image/png' if ext == '.png' else 'image/jpeg'
                content_parts.append({'type': 'image_url',
                    'image_url': {'url': f'data:{mime};base64,{data_ref}'}})
            except Exception as e:
                print(f'No se pudo cargar {ref_path}: {e}')
        features_lista = '\n'.join(f'  - {f}' for f in FEATURES_DISCRIMINATORIAS)
        content_parts.append({'type': 'text', 'text': (
            '\n=== INSTRUCCIONES ===\n\n'
            'Compara imagen de consulta vs cada referencia en estas features:\n'
            f'{features_lista}\n\n'
            'Proporciona:\n'
            '1. TABLA COMPARATIVA (Markdown)\n'
            '2. VEREDICTO POR REFERENCIA\n'
            '3. CONCLUSION FINAL con confianza diagnostica'
        )})
        try:
            resp = await self.llm.ainvoke([HumanMessage(content=content_parts)])
            state['analisis_comparativo'] = resp.content
            estructura = await self._extraer_estructura_de_analisis(resp.content)
            state['estructura_identificada'] = estructura
        except Exception as e:
            print(f'Error en analisis comparativo: {e}')
            state['analisis_comparativo'] = None
        state['trayectoria'].append({'nodo': 'AnalisisComparativo',
            'refs': len(imagenes_referencia), 'tiempo': round(time.time()-t0,2)})
        return state

    async def _extraer_estructura_de_analisis(self, analisis):
        try:
            resp = await self.llm.ainvoke([
                SystemMessage(content=(
                    'Extrae el nombre de la estructura histologica mas probable del analisis. '
                    'Responde SOLO con el nombre. Si no se puede determinar: indeterminado.')),
                HumanMessage(content=analisis[-1000:])
            ])
            return resp.content.strip()
        except:
            return None

    async def _nodo_generar_respuesta(self, state):
        t0 = time.time()
        if not state['contexto_suficiente']:
            if state.get('tiene_imagen') and state.get('imagen_path'):
                state['respuesta_final'] = (
                    'No encontre fragmentos relevantes en la base de datos, '
                    'pero puedo ofrecerte el analisis visual de la imagen:\n\n'
                    f'{_safe(state.get("analisis_visual"), "No disponible")}'
                )
                state['contexto_suficiente'] = True
            else:
                state['respuesta_final'] = (
                    f'No encontre informacion suficientemente relevante '
                    f'(umbral {self.SIMILARITY_THRESHOLD:.0%} no alcanzado).\n'
                    f'Consulta: {state["consulta_busqueda_texto"]}'
                )
            state['trayectoria'].append({'nodo': 'GenerarRespuesta',
                'contexto_suficiente': False, 'tiempo': round(time.time()-t0,2)})
            return state
        tiene_comparativo = bool(_safe(state.get('analisis_comparativo')))
        nota_comparativa  = (
            '\n\nIMPORTANTE: Ya existe un ANALISIS COMPARATIVO DETALLADO. '
            'Incorporalo con PRIORIDAD en el diagnostico diferencial.'
        ) if tiene_comparativo else ''
        system_prompt = (
            'Eres un patologo experto en histologia.\n\n'
            'Responde basandote en (1) FRAGMENTOS DE REFERENCIA, '
            '(2) IMAGENES DE REFERENCIA y (3) ANALISIS COMPARATIVO.\n\n'
            'Estructura:\n'
            '1. Descripcion de la imagen del usuario\n'
            '2. Estructuras identificadas [Fuente: archivo]\n'
            '3. Analisis comparativo (si aplica)\n'
            '4. Diagnostico diferencial con diferencias morfologicas clave\n'
            f'5. Conclusion y confianza diagnostica{nota_comparativa}'
        )
        content_parts = [{'type': 'text', 'text': (
            f'CONSULTA: {state["consulta_texto"]}\n\n'
            f'HISTORIAL: {_safe(state.get("contexto_memoria"))[:300]}\n\n'
            f'ANALISIS VISUAL:\n{_safe(state.get("analisis_visual"),"No disponible")[:800]}\n\n'
            f'FRAGMENTOS DE REFERENCIA:\n{state["contexto_documentos"]}'
            + (f'\n\nANALISIS COMPARATIVO:\n{state["analisis_comparativo"][:2000]}'
               if _safe(state.get('analisis_comparativo')) else '')
            + '\n\nResponde integrando todos los elementos anteriores.'
        )}]
        imagen_path = state.get('imagen_path')
        if state.get('tiene_imagen') and imagen_path and os.path.exists(imagen_path):
            try:
                with open(imagen_path, 'rb') as f:
                    data = base64.b64encode(f.read()).decode('utf-8')
                ext  = os.path.splitext(imagen_path)[1].lower()
                mime = 'image/png' if ext == '.png' else 'image/jpeg'
                label = 'NUEVA IMAGEN' if state.get('imagen_es_nueva') \
                        else f'IMAGEN ACTIVA (turno {self.memoria.imagen_turno_subida})'
                content_parts.append({'type': 'text', 'text': f'\n**{label}:**'})
                content_parts.append({'type': 'image_url',
                    'image_url': {'url': f'data:{mime};base64,{data}'}})
            except Exception as e:
                print(f'No se pudo agregar imagen: {e}')
        imagenes_usadas = 0
        for img_path in state.get('imagenes_recuperadas',[])[:3]:
            if not os.path.exists(img_path):
                continue
            try:
                with open(img_path, 'rb') as f:
                    data = base64.b64encode(f.read()).decode('utf-8')
                ext  = os.path.splitext(img_path)[1].lower()
                mime = 'image/png' if ext == '.png' else 'image/jpeg'
                content_parts.append({'type': 'text',
                    'text': f'\n**REFERENCIA [{os.path.basename(img_path)}]:**'})
                content_parts.append({'type': 'image_url',
                    'image_url': {'url': f'data:{mime};base64,{data}'}})
                imagenes_usadas += 1
            except Exception as e:
                print(f'No se pudo cargar {img_path}: {e}')
        try:
            resp = await self.llm.ainvoke([
                SystemMessage(content=system_prompt),
                HumanMessage(content=content_parts)
            ])
            state['respuesta_final'] = resp.content
        except Exception as e:
            state['respuesta_final'] = f'Error generando respuesta: {e}'
        state['trayectoria'].append({'nodo': 'GenerarRespuesta',
            'contexto_suficiente': True, 'imagenes_usadas': imagenes_usadas,
            'tiempo': round(time.time()-t0,2)})
        return state

    async def _nodo_finalizar(self, state):
        if state.get('respuesta_final'):
            self.memoria.add_interaction(state['consulta_texto'],
                                          state['respuesta_final'])
        total = round(time.time() - state['tiempo_inicio'], 2)
        state['trayectoria'].append({'nodo': 'Finalizar', 'tiempo_total': total})
        with open('trayectoria_histologia.json', 'w', encoding='utf-8') as f:
            json.dump({'trayectoria': state['trayectoria'],
                       'estructura_identificada': state.get('estructura_identificada'),
                       'imagenes_recuperadas': state.get('imagenes_recuperadas',[])},
                      f, indent=4, ensure_ascii=False)
        print(f'Flujo completado en {total}s')
        return state

    # ── Embeddings ────────────────────────────────────────────────────────
    def _embed_texto(self, texto):
        inputs = self.clip_tokenizer(
            [texto], padding=True, truncation=True,
            max_length=self.max_token_length, return_tensors='pt'
        ).to(self.device)
        with torch.no_grad():
            feats = self.clip_model.get_text_features(**inputs)
            emb   = feats.pooler_output.cpu().numpy().flatten() \
                    if hasattr(feats,'pooler_output') \
                    else feats.cpu().numpy().flatten()
        return emb / np.linalg.norm(emb)

    def _embed_imagen_pil(self, pil_img):
        """Recibe PIL.Image, devuelve numpy array normalizado (768d)."""
        tensor = self.conch_transform(pil_img).unsqueeze(0).to(self.device)
        with torch.no_grad():
            emb = self.conch_model(tensor)
            if isinstance(emb, (list, tuple)):
                emb = emb[0]
            emb = emb.cpu().numpy().flatten()
        return emb / np.linalg.norm(emb)

    def _embed_imagen(self, imagen_path):
        """Recibe path, devuelve numpy array normalizado (768d)."""
        img = Image.open(imagen_path).convert('RGB')
        return self._embed_imagen_pil(img)

    def _leer_pdf(self, path):
        try:
            reader = PdfReader(path)
            return ''.join(p.extract_text() or '' for p in reader.pages)
        except Exception as e:
            print(f'Error leyendo {path}: {e}')
            return ''

    def _chunks(self, texto, size=CHUNK_SIZE):
        return [texto[i:i+size] for i in range(0, len(texto), size)]

    # ── Qdrant ────────────────────────────────────────────────────────────
    async def _get_qdrant_client(self):
        return AsyncQdrantClient(url=self.qdrant_url, api_key=self.qdrant_api_key)

    async def _search_qdrant(self, texto=None, imagen_embedding=None,
                              top_k=10, score_threshold=0.0):
        try:
            client = await self._get_qdrant_client()
            await client.get_collection(self.collection_name)
        except Exception as e:
            print(f'Qdrant no disponible: {e}')
            return []

        if imagen_embedding:
            query_vec = imagen_embedding
            using     = 'image'
        elif texto:
            query_vec = self._embed_texto(texto).tolist()
            using     = 'text'
        else:
            return []

        results = await client.query_points(
            collection_name=self.collection_name,
            query=query_vec,
            using=using,
            limit=top_k,
            with_payload=True,
            with_vectors=False,
            score_threshold=score_threshold
        )
        return [{
            'tipo':        (p.payload or {}).get('tipo', 'desconocido'),
            'fuente':      (p.payload or {}).get('fuente', 'N/A'),
            'texto':       (p.payload or {}).get('texto', ''),
            'imagen_path': (p.payload or {}).get('imagen_path'),
            'similitud':   round(p.score, 4)
        } for p in results.points]

    # ── Indexacion ────────────────────────────────────────────────────────
    def procesar_contenido_base(self, directorio=DIRECTORIO_PDFS):
        pdfs = glob.glob(os.path.join(directorio, '*.pdf'))
        if not pdfs:
            print(f'Sin PDFs en {directorio}')
            return ''
        self.contenido_base = '\n'.join(self._leer_pdf(p) for p in pdfs)
        print(f'{len(pdfs)} PDFs leidos ({len(self.contenido_base)} chars)')
        return self.contenido_base[:500]


    async def extraer_y_preparar_temario(self):
        if not self.contenido_base:
            print('Contenido base vacio')
            return
        await self.extractor_temario.extraer_temario(self.contenido_base)
        if self.clasificador_semantico:
            self.clasificador_semantico.temario = self.extractor_temario.temas

    async def indexar_en_qdrant(self, directorio_pdfs=DIRECTORIO_PDFS,
                                imagen_files_extra=None):
        points    = []
        global_id = 0

        print('Indexando texto de PDFs...')
        for pdf_path in glob.glob(os.path.join(directorio_pdfs, '*.pdf')):
            texto = self._leer_pdf(pdf_path)
            for i, chunk in enumerate(self._chunks(texto)):
                try:
                    emb = self._embed_texto(chunk)
                    points.append(PointStruct(
                        id=global_id,
                        vector={'text': emb.tolist()},
                        payload={'tipo': 'texto',
                                'fuente': os.path.basename(pdf_path),
                                'chunk_id': i, 'texto': chunk}
                    ))
                    global_id += 1
                except Exception as e:
                    print(f'Chunk {i}: {e}')

        print('Extrayendo e indexando imagenes...')
        imagenes_pdf = self.extractor_imagenes.extraer_de_directorio(directorio_pdfs)
        for img_info in imagenes_pdf:
            img_path = img_info['path']
            if not os.path.exists(img_path):
                continue
            try:
                emb = self._embed_imagen(img_path)
                descripcion = (
                    f'Imagen histologica extraida de {img_info["fuente_pdf"]}, '
                    f'pagina {img_info["pagina"]}'
                )
                if img_info.get('ocr_text'):
                    descripcion += f'. Anotaciones: {img_info["ocr_text"]}'
                points.append(PointStruct(
                    id=global_id,
                    vector={'image': emb.tolist()},
                    payload={'tipo': 'imagen',
                            'fuente': img_info['fuente_pdf'],
                            'pagina': img_info['pagina'],
                            'chunk_id': img_info['indice'],
                            'texto': descripcion,
                            'imagen_path': img_path,
                            'ocr_text': img_info.get('ocr_text', '')}
                ))
                global_id += 1
            except Exception as e:
                print(f'Error {img_path}: {e}')

        if not points:
            print('No hay puntos para indexar')
            return

        client = await self._get_qdrant_client()
        try:
            await client.get_collection(self.collection_name)
            print(f'Coleccion {self.collection_name} ya existe — usando la existente')
        except:
            await client.create_collection(
                collection_name=self.collection_name,
                vectors_config={
                    'text':  VectorParams(size=512,  distance=Distance.COSINE),
                    'image': VectorParams(size=768,  distance=Distance.COSINE),
                }
            )
            print(f'Coleccion {self.collection_name} creada (text:512d, image:768d)')

        total = 0
        for i in range(0, len(points), 100):
            lote = points[i:i+100]
            await client.upsert(collection_name=self.collection_name, points=lote)
            total += len(lote)
            print(f'  Lote {i//100+1}: {len(lote)} pts ({total}/{len(points)})')

        n_t = sum(1 for p in points if 'text'  in p.vector)
        n_i = sum(1 for p in points if 'image' in p.vector)
        print(f'{len(points)} indexados ({n_t} texto, {n_i} imagen)')


    # ── Consulta publica ──────────────────────────────────────────────────
    async def consultar(self, consulta_texto, imagen_path=None,
                         user_id='default_user'):
        imagen_activa = imagen_path or self.memoria.get_imagen_activa()
        print(f'\n{"="*60}')
        print(f'RAG Multimodal v4.2 | umbral={self.SIMILARITY_THRESHOLD}')
        print(f'  Texto : {consulta_texto}')
        print(f'  Imagen: {imagen_activa or "ninguna"}')
        print(f'{"="*60}')
        initial_state = AgentState(
            messages=[], consulta_texto=consulta_texto,
            imagen_path=imagen_path, imagen_embedding=None,
            contexto_memoria='', contenido_base=self.contenido_base,
            clasificacion='', consulta_busqueda_texto='',
            consulta_busqueda_visual='', resultados_busqueda=[],
            resultados_validos=[], contexto_documentos='',
            respuesta_final='', trayectoria=[], user_id=user_id,
            tiempo_inicio=time.time(), analisis_visual=None,
            tiene_imagen=False, imagen_es_nueva=False,
            contexto_suficiente=False, temario=self.extractor_temario.temas,
            tema_valido=True, tema_encontrado=None,
            imagenes_recuperadas=[], analisis_comparativo=None,
            estructura_identificada=None, similitud_semantica_dominio=0.0,
        )
        config = {'configurable': {'thread_id': user_id},
                  'run_name': f'consulta-v4.2-{user_id}',
                  'tags': ['rag','histologia','multimodal','v4.2']}
        try:
            final     = await self.compiled_graph.ainvoke(initial_state, config=config)
            respuesta = final['respuesta_final']
        except Exception as e:
            import traceback; traceback.print_exc()
            respuesta = f'Error en el flujo: {e}'
        print(f'\n{"="*60}\nRESPUESTA FINAL:\n{"="*60}')
        print(respuesta)
        print('='*60)
        return respuesta


print('AsistenteHistologiaMultimodal definido')

AsistenteHistologiaMultimodal definido


## Celda 7 — Inicializar componentes
> Descarga CLIP (~600MB). Tarda 1-2 minutos la primera vez.
> Si Colab desconecta, volver a correr desde aqui (no desde celda 1).

In [8]:
os.makedirs(DIRECTORIO_IMAGENES, exist_ok=True)
os.makedirs(DIRECTORIO_PDFS,     exist_ok=True)

asistente = AsistenteHistologiaMultimodal()
asistente.inicializar_componentes()
print('Listo para procesar PDFs')

AsistenteHistologiaMultimodal v4.2 creado
LLM inicializado (Gemini 2.5 Flash)
Cargando CLIP...


config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

CLIP en cuda
Cargando CONCH...


pytorch_model.bin:   0%|          | 0.00/802M [00:00<?, ?B/s]

  Checkpoint OK | keys totales: 326
  Missing: 150 | Unexpected: 174
  Smoke test OK | dim=768 | norma=1.0
CONCH en cuda
Todos los componentes inicializados
Listo para procesar PDFs


## Celda 8 — Procesar PDFs y extraer temario
> Requiere que el PDF este en `/content/pdf/`.
> Subir el archivo antes de correr esta celda.

In [9]:
import glob as _glob

pdfs_disponibles = _glob.glob(f'{DIRECTORIO_PDFS}/*.pdf')
if not pdfs_disponibles:
    raise FileNotFoundError(
        f'No hay PDFs en {DIRECTORIO_PDFS}. '
        'Subi el manual antes de continuar.'
    )
print(f'PDFs encontrados: {pdfs_disponibles}')

asistente.procesar_contenido_base(DIRECTORIO_PDFS)
await asistente.extraer_y_preparar_temario()
print(f'Temario listo: {len(asistente.extractor_temario.temas)} temas')
print('Muestra:', asistente.extractor_temario.temas[:5])

PDFs encontrados: ['/content/pdf/arch2.pdf', '/content/pdf/arch3.pdf', '/content/pdf/arch4.pdf']
3 PDFs leidos (32249 chars)
Extrayendo temario con LLM...
Temario: 78 temas
Temario listo: 78 temas
Muestra: ['Pericondrio', 'Capa fibrosa del pericondrio', 'Fibroblastos', 'Fibras de colágena', 'Capa condrógena del pericondrio']


## Celda 9 — Indexar en Qdrant
> **Saltear esta celda** si Qdrant ya tiene los datos de una sesion anterior.
> Solo correr cuando el PDF cambia o es la primera vez.

In [12]:
# Verificar si ya existe la coleccion antes de reindexar
from qdrant_client import QdrantClient as _QC
_qc = _QC(url=asistente.qdrant_url, api_key=asistente.qdrant_api_key)
try:
    info = _qc.get_collection(asistente.collection_name)
    n_puntos = info.points_count
    print(f'Coleccion "{asistente.collection_name}" ya existe con {n_puntos} puntos.')
    print('Si queres reindexar igualmente, comenta el try/except y descomenta las ultimas 3 lineas.')
    print('Saltando indexacion.')
except:
    print('Coleccion no existe. Indexando...')
    await asistente.indexar_en_qdrant(DIRECTORIO_PDFS)
    print('Indexacion completa')

# Esto hace que siempre indexe, independientemente de si la colección ya existe.
# La función indexar_en_qdrant internamente usa upsert, así que si la colección
# ya tiene datos los va a sobreescribir sin borrar la colección primero.
#print('Indexando...')
#await asistente.indexar_en_qdrant(DIRECTORIO_PDFS)
#print('Indexacion completa')

Coleccion "histologia_v4_multimodal" ya existe con 124 puntos.
Si queres reindexar igualmente, comenta el try/except y descomenta las ultimas 3 lineas.
Saltando indexacion.


## Celda 10 — Consulta interactiva
Correr esta celda cada vez que quieras hacer una consulta.

**Comandos especiales:** `temario`, `imagen actual`, `nueva imagen`, `salir`

In [13]:
async def modo_interactivo():
    print('''
RAG Histologia v4.2 - Chat Interactivo
---------------------------------------
  Escribe tu pregunta y presiona Enter
  Para subir una imagen: escribe el PATH cuando se pida
  La imagen se recuerda entre turnos
  Comandos: temario | imagen actual | nueva imagen | salir
''')
    while True:
        try:
            img_activa = asistente.memoria.get_imagen_activa()
            if img_activa:
                print(f'[Imagen activa: {os.path.basename(img_activa)} '
                      f'(turno {asistente.memoria.imagen_turno_subida})]')
            consulta = input('Vos: ').strip()
            if not consulta:
                continue
            cmd = consulta.lower()
            if cmd in ('salir', 'exit', 'quit'):
                print('Hasta luego!')
                break
            if cmd == 'temario':
                for i, t in enumerate(asistente.extractor_temario.temas, 1):
                    print(f'  {i:3}. {t}')
                continue
            if cmd == 'imagen actual':
                print(f'Imagen activa: {img_activa or "ninguna"}')
                continue
            if cmd == 'nueva imagen':
                asistente.memoria.set_imagen(None)
                print('Imagen activa eliminada.')
                continue
            imagen_path = None
            img_input   = input('Imagen (path o Enter para omitir): ').strip()
            if img_input:
                if os.path.exists(img_input):
                    imagen_path = img_input
                    print(f'Nueva imagen: {imagen_path}')
                else:
                    print(f'No encontrada: {img_input} - se usara imagen activa')
            await asistente.consultar(consulta, imagen_path)
        except KeyboardInterrupt:
            print('\nInterrumpido')
            break
        except Exception as e:
            import traceback; traceback.print_exc()
            print(f'Error: {e}')

await modo_interactivo()


RAG Histologia v4.2 - Chat Interactivo
---------------------------------------
  Escribe tu pregunta y presiona Enter
  Para subir una imagen: escribe el PATH cuando se pida
  La imagen se recuerda entre turnos
  Comandos: temario | imagen actual | nueva imagen | salir

Vos: salir
Hasta luego!
